In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error
from itertools import product
import warnings
warnings.filterwarnings("ignore")

path = "../OIWORLD.csv"  # Run net_trade.ipynb from analysis/oil_info/
oiworld = pd.read_csv(path)

# OIWORLD.csv column names: Region, Category, Product, Year, Unit, Value, Flag
# ALL CAPS region names — net trade covers both importers AND major exporters
products = ["OIL_PRIM_PRODS_X", "OIL_SEC_PRODUCTS"]

# These are the exact Region values in OIWORLD.csv for the 20 net trade countries
nt_countries_oiworld = [
    "BRAZIL", "CANADA", "CHINA", "FRANCE", "GERMANY", "INDIA",
    "IRAN", "IRAQ", "JAPAN", "KAZAKHSTA", "KOREA", "KUWAIT",
    "NORWAY", "QATAR", "RUSSIA", "SAUDIARAB", "SINGAPORE",
    "SPAIN", "UAE", "USA"
]

df_exp = oiworld[
    (oiworld["Product"].isin(products)) &
    (oiworld["Category"] == "EXPORTS") &
    (oiworld["Unit"] == "KBD") &
    (oiworld["Region"].isin(nt_countries_oiworld)) &
    (oiworld["Year"] <= 2023)  # cap at 2023 so steps=7 gives 2024–2030
]
df_exp = df_exp.groupby(["Region", "Year"], as_index=False)["Value"].sum()

df_imp = oiworld[
    (oiworld["Product"].isin(products)) &
    (oiworld["Category"] == "IMPORTS") &
    (oiworld["Unit"] == "KBD") &
    (oiworld["Region"].isin(nt_countries_oiworld)) &
    (oiworld["Year"] <= 2023)
]
df_imp = df_imp.groupby(["Region", "Year"], as_index=False)["Value"].sum()

merged = df_exp.merge(df_imp, on=["Region", "Year"], how="outer", suffixes=("_exp", "_imp")).fillna(0)
merged["Net_Trade"] = merged["Value_exp"] - merged["Value_imp"]
merged = merged.rename(columns={"Region": "Country"})

# Normalise country names (ALL CAPS → proper case)
merged["Country"] = merged["Country"].str.title()
COUNTRY_FIXES = {
    "Usa": "USA", "Uae": "UAE",
    "Saudiarab": "Saudi Arabia", "Kazakhsta": "Kazakhstan",
}
merged["Country"] = merged["Country"].replace(COUNTRY_FIXES)

In [2]:
# ── Time-series helpers ───────────────────────────────────────────────────
# Value > 0 filter is required: Russia and Kazakhstan have zeros 1971–1989
# (Soviet-era — country-level data didn't exist); Kuwait imports are zero
# 1971–2016 (only 7 nonzero years: 2017–2023); Iran, Iraq, Saudi Arabia,
# Qatar, and Korea each have 1–4 isolated zero years. np.log(0) = -inf
# crashes the ARIMA grid search, silently dropping those countries.
# The merged df (used for historical rows) is NOT filtered — raw zero rows
# are kept there and the frontend's fetchNetTrade already filters them out.
def build_ts_exp(region):
    df_c = df_exp[(df_exp["Region"] == region) & (df_exp["Value"] > 0)].sort_values("Year")
    ts = df_c.set_index(pd.to_datetime(df_c["Year"], format="%Y"))["Value"]
    return ts.asfreq("YS")

def build_ts_imp(region):
    df_c = df_imp[(df_imp["Region"] == region) & (df_imp["Value"] > 0)].sort_values("Year")
    ts = df_c.set_index(pd.to_datetime(df_c["Year"], format="%Y"))["Value"]
    return ts.asfreq("YS")

# ── Grid search: best ARIMA order per country, per series ─────────────────
best_models_exp = {}
best_models_imp = {}

for region in nt_countries_oiworld:
    for label, build_fn, best_models in [
        ("Exp", build_ts_exp, best_models_exp),
        ("Imp", build_ts_imp, best_models_imp),
    ]:
        try:
            log_ts = np.log(build_fn(region))
            best_aic, best_order = np.inf, (1, 1, 1)
            for p, d, q in product(range(4), range(2), range(4)):
                try:
                    fit = ARIMA(log_ts, order=(p, d, q)).fit()
                    if fit.aic < best_aic:
                        best_aic, best_order = fit.aic, (p, d, q)
                except Exception:
                    continue
            best_models[region] = best_order
            print(f"{region} {label}: {best_order}, AIC={best_aic:.1f}")
        except Exception as e:
            print(f"WARNING: {region} {label} grid search failed — {e}")

# ── Backtest + forecast ───────────────────────────────────────────────────
exp_results = []
imp_results = []

for region in nt_countries_oiworld:
    # Normalise ALL CAPS key → display name so the export code's
    # merged[merged["Country"] == country] lookup finds the right rows.
    country_norm = COUNTRY_FIXES.get(region.title(), region.title())
    for label, build_fn, best_models, results in [
        ("Exp", build_ts_exp, best_models_exp, exp_results),
        ("Imp", build_ts_imp, best_models_imp, imp_results),
    ]:
        order = best_models.get(region, (1, 1, 1))
        try:
            ts = build_fn(region)
            log_ts = np.log(ts)
            mape_val = ""
            try:
                train_log, test = log_ts[:-5], ts[-5:]
                backtest_fc = np.exp(ARIMA(train_log, order=order).fit().forecast(steps=5))
                mape_val = round(mean_absolute_percentage_error(test, backtest_fc) * 100, 1)
            except Exception:
                pass
            fit = ARIMA(log_ts, order=order).fit()
            fc_obj = fit.get_forecast(steps=7)  # steps=7 → 2024–2030
            results.append({
                "Country": country_norm, "Order": order, "ts": ts,
                "forecast": np.exp(fc_obj.predicted_mean),
                "conf_int": np.exp(fc_obj.conf_int()),
                "MAPE": mape_val,
            })
        except Exception as e:
            print(f"WARNING: ARIMA failed for {region} {label} — {e}")

print(f"exp_results: {len(exp_results)}/{len(nt_countries_oiworld)} fit")
print(f"imp_results: {len(imp_results)}/{len(nt_countries_oiworld)} fit")

BRAZIL Exp: (0, 1, 1), AIC=35.7


BRAZIL Imp: (2, 0, 3), AIC=-85.6


CANADA Exp: (1, 1, 0), AIC=-91.1


CANADA Imp: (1, 1, 0), AIC=-99.1


CHINA Exp: (1, 1, 0), AIC=-8.0


...

UAE Exp: (0, 1, 3), AIC=-105.7


UAE Imp: (1, 1, 0), AIC=-8.3


USA Exp: (1, 1, 3), AIC=-80.2


USA Imp: (1, 1, 0), AIC=-104.0


exp_results: 20/20 fit
imp_results: 20/20 fit


In [3]:
exp_by_country = {r["Country"]: r for r in exp_results}
imp_by_country = {r["Country"]: r for r in imp_results}
nt_rows = []

for country in exp_by_country:
    if country not in imp_by_country:
        print(f"WARNING: skipping {country} — imports ARIMA failed, no net trade row written")
        continue
    exp_r = exp_by_country[country]
    imp_r = imp_by_country[country]
    exp_order = str(exp_r["Order"])
    imp_order = str(imp_r["Order"])
    exp_mape = exp_r.get("MAPE", "")
    imp_mape = imp_r.get("MAPE", "")
    if exp_mape != "" and imp_mape != "":
        avg_mape = round((float(exp_mape) + float(imp_mape)) / 2, 1)
    else:
        avg_mape = ""  # blank so Step 6 'populated' filter flags it as missing

    # Historical rows — use raw merged values (not modelled)
    hist = merged[merged["Country"] == country].sort_values("Year")
    for _, row in hist.iterrows():
        nt_rows.append({
            "Country": country, "Year": int(row["Year"]), "Type": "Historical",
            "Net_Trade": round(row["Net_Trade"], 1),
            "Net_CI_Low": "", "Net_CI_High": "",
            "Exports": round(row["Value_exp"], 1), "Imports": round(row["Value_imp"], 1),
            "Exports_Order": exp_order, "Imports_Order": imp_order, "Avg_MAPE": "",
        })

    # Forecast rows — combine the two models with worst-case CI propagation
    exp_fc, exp_ci = exp_r["forecast"], exp_r["conf_int"]
    imp_fc, imp_ci = imp_r["forecast"], imp_r["conf_int"]
    for i, dt in enumerate(exp_fc.index):
        e_val = float(exp_fc.iloc[i])
        i_val = float(imp_fc.iloc[i])
        nt_rows.append({
            "Country": country, "Year": dt.year, "Type": "Forecast",
            "Net_Trade": round(e_val - i_val, 1),
            "Net_CI_Low":  round(float(exp_ci.iloc[i, 0]) - float(imp_ci.iloc[i, 1]), 1),
            "Net_CI_High": round(float(exp_ci.iloc[i, 1]) - float(imp_ci.iloc[i, 0]), 1),
            "Exports": round(e_val, 1), "Imports": round(i_val, 1),
            "Exports_Order": exp_order, "Imports_Order": imp_order, "Avg_MAPE": avg_mape,
        })

pd.DataFrame(nt_rows).to_csv("net_trade_forecast.csv", index=False)
print(f"Saved net_trade_forecast.csv — {len(nt_rows)} rows")

Saved net_trade_forecast.csv — 1200 rows
